In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

project = os.getenv('GOOGLE_CLOUD_PROJECT')
print(project)  # DEBUG

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # safer model
    vertexai=True,
    project=project
)

munna-genai


In [29]:
# Lets create a  simple Tool  to greet a person
from langchain.tools import tool

@tool
def say_hello(name:str) -> str:
    """This function is used to greet a person

    Args:
        name (str): name

    Returns:
        str: Greeting
    """
    return f"Hello {name}"

In [4]:
llm_with_tools = llm.bind_tools([say_hello])

In [5]:
# llm which is not aware of tools
response = llm.invoke("Greet shyam")
response

AIMessage(content='Hello Shyam!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d58f7-2850-7873-ab31-fb28f62d798c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 22, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 19}})

In [6]:
response = llm_with_tools.invoke("Greet shyam")
response

AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}, '__gemini_function_call_thought_signatures__': {'4889bdfe-719f-436e-845c-2429bbcf8c71': 'CvABAY89a1/ecrRznoEf3n/Vw/jJC3XdS5+LotMUCgfnjxMXqasM4WfpHN16jMUXkq5Q9BOcfQCGShO2QlQpO+NEDQolgcYV07yRVoygwp4c+FAvyrcB6Hnu/wMvZK2fSaltLA+ZnQP1RAEeBcGzi7zWZACzhgspoLHwW8NlmzxSkNF+8mcsmLLa4jyHI/6oM5Br9QGypUVjkCqUMIhQ5HqJiriL9sv4kZegFzaaLZWFZvw3eNiG5YBTGNqOh20xy/fl0uD2IxF/33GIorkCrwhMr8entEPY9pGoe1izQdQG49HvRnERdR0lqHKPPWXDR2oJ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d58f7-5726-75a3-b43d-0c23a9593189-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': '4889bdfe-719f-436e-845c-2429bbcf8c71', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 63, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}, 'output

In [7]:
from langchain.agents import create_agent
# create a agent
agent = create_agent(
    model=llm,
    tools=[say_hello]
)

In [8]:
agent_response = agent.invoke({
    "messages": [
        ("user", "Greet shyam")
    ]
})

In [9]:
agent_response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Hello shyam


In [10]:
agent_response['messages']

[HumanMessage(content='Greet shyam', additional_kwargs={}, response_metadata={}, id='ddc4af23-5057-4cd9-93bc-84d1b919562a'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}, '__gemini_function_call_thought_signatures__': {'55a3d410-570b-4225-9d4b-725fc8e59ab0': 'CvsBAY89a196lcs2NSe0mFvuxuNp+f9EARucyIYWZsPgL6qChvstg3I+sjAhNf+LUoGFJTlig2cVHuYJw6mn/ao7fWfFMC+wYL328JDmNj46UjPYRXDrYbGsyDp4xSSd6Urbb4+hPBNpKIDfgc08ve9vU5QFyZSNHjI/CQw9HiBKz/dLoHfqXCBI5FhC+K0mUdMl477vi4s2INEM9XyovD6ZIL0ziB8II4UZLtP98yU6aZ7/MiZuT97SJ+4yj8lmjAigCJrHE1U+n6jVEY449t4xbpCm0BbGlt1Mp+p8Z19OfG96/2RQVchm3issPl5UpPGdSkWH3Cx5YC3GUjM='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d58f7-c8eb-7de1-8d6b-03538d9869de-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': '55a3d410-570b-4225-9d4b-725fc8e59ab0', 'type': 'tool_call'}], invalid

In [12]:
# Create a Tool using Tavily Search API
## I want to search internet
load_dotenv()
#os.getenv('TAVILY_API_KEY')

True

In [13]:
# lets create tavily search tool
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results = 3,
    topic = "news"
)

In [14]:
# lets create an agent with tavily search tool

agent = create_agent(
    model=llm,
    tools=[tavily_search_tool]
)

In [15]:
response = agent.invoke({
    "messages": "Get me latest news about gpu's "
})

In [16]:
response['messages']

[HumanMessage(content="Get me latest news about gpu's ", additional_kwargs={}, response_metadata={}, id='10d17c9c-41ab-4b27-be03-e6d96930525b'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "latest news about GPUs", "topic": "news"}'}, '__gemini_function_call_thought_signatures__': {'48786400-8e6a-40cb-8525-5941e8f6e6cd': 'Co8CAY89a18hpRhZ2MRkQy78wan8UxMEc8NIA/aziqt+44pCnLPMabI/WMtKBQ3b6wbaSN6B9yDmaW+obe9DSQjgxC4X5AvtHUdU8uvfk1yT3p6rHK1gmBs+PxSAY22taDFD7r849Thyz7O94Mj0QWlh0IWw4Y5c1ZxEUprSq8rmRp2w+q4mbejNrHS/UaxOhrBlLchvhbDkXL4RjMjF0kpPRisT9CCHnoLS+ZvqI7Zn94z85/1OepKa859UDdf6joVhWTL7pzd1M0GDQNFODQB/LWTbaqv7L5P5laJEIyRhJGzs8xavQTkLKdWLocoSMlEUSZTUpGDi/dEme1YhXrzR0rsIzTyz8dPFp5T+n7qdbw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d58fc-218f-79c3-902d-8377a5d56ca1-0', tool_calls=[{'name': 'tavily_search', 'args': {'qu

In [17]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': "Here's the latest news about GPUs:\n\n*   **New Rowhammer Attacks:** Modern GPUs, including the GeForce RTX 3060 and RTX A6000, have been found vulnerable to new Rowhammer attacks.\n*   **Nvidia H100 Demand and Price Surge:** The demand for Nvidia's H100 GPUs has led to a 40% increase in rental prices, now at about $2.35 per hour.\n*   **RTX 3080 Cooling Mod:** A YouTuber successfully modified a GeForce RTX 3080 with a workstation AIO cooler, which reportedly halved its VRAM temperature.", 'extras': {'signature': 'CpgFAY89a198IhI8cSU1dbC0oEGCN5U8Jpoj4OQv/6Kk4MpJnwg/1TQsSWpo9eyr8ejlwxVpUMZ8j3UNxkQK2m1yR0niKyfPP9dHpkIIoN3IfK7bnmffT4jsYUiUeu1Kmzt59vIOf+JuR40FazqZ5EhlO4iVZPA+eknaoI90V3N9eMLhJfgCxkwxWZK9F2Eyn/3ANPWIMVBfE0Wu5dRhqldPkbOpLX3vBPNtQS+g4KYj3Q7j1IGEgHCpnJm9afeLPN90TUEsV3yjEJ4mVBAB5t/ye0Aaj5cytia5OIt/WsNXh+bE1TdpN1fhENyEi08MPWy850a70aNdPJ0GTIO6a5Igxv9CaEdUvLzMz+1sH4T2sZauY0e

In [18]:
# Create a Tool to find friends of a person
@tool
def find_friends(name:str) -> list:
    """Find friends of a person 
    """
    return ["Amar", "Akbar", "Anthony"]

In [19]:
agent = create_agent(
    model=llm,
    tools = [find_friends]
)

In [20]:
response = agent.invoke({
    "messages": "Find friends of Ram"
})

In [21]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Ram's friends are Amar, Akbar, and Anthony.


In [24]:
response['messages']

[HumanMessage(content='Find friends of Ram', additional_kwargs={}, response_metadata={}, id='37a78180-83cc-4449-a8f3-37b386d01792'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'find_friends', 'arguments': '{"name": "Ram"}'}, '__gemini_function_call_thought_signatures__': {'df354d11-d060-44be-baf0-c28dd1c4b7ca': 'CtkBAY89a1/SUI1rcPkEWDJeNaXBaaJN+Rmb+4aIyOPviFDBbG7uw1YLOyikhkC/WXF3KgDq0D9aOOixKiD+n+vzpZaibqkrxO5V/qMwws5fSz2MmgBV7jdi1lxAiiXEY/u9m5i25zmZmYeHx1zog1QV8Y6hTsttNojHYzV0qvtfAD7NOtF3oCHsiUWVtO5QvCPX2HOh/DkDuxGJ6WU4H5MyCnBhWPuCp7sRlfScNC0xEBpS3WkEpoeNQCNq6nShshZjiC6gsF1TDneGnL19d5em/Y0V7Rg9YLRIpA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d58fd-9932-7612-8983-569993e906d1-0', tool_calls=[{'name': 'find_friends', 'args': {'name': 'Ram'}, 'id': 'df354d11-d060-44be-baf0-c28dd1c4b7ca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i

In [30]:
# Combining the Tools
agent = create_agent(
    model=llm,
    tools = [find_friends, say_hello]
)

In [31]:
response = agent.invoke({
    "messages": "Find friends of Ram and ensure you greet each friend"
})

In [33]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Find friends of Ram and ensure you greet each friend
================================== Ai Message ==================================

[{'type': 'text', 'text': "I can find friends of Ram, but I cannot greet each of them individually using the available tools. Would you like me to proceed with just finding Ram's friends?", 'extras': {'signature': 'CpUHAY89a1/lPyLIhHiAsODgc99savVE8Kf1JbOiZovhMNIblNQcXwpvD2Lg9Bo3ExDPAZeBCsHCtMIM4S7adMy7zPNZk9EejSW/odX6KJZ+1CLz9ofnWeQWdrRm5t9TEJOtidmR3GBXV0ZGeuhXRf0MiThpweLtyoy6vg4LOuDusMzOzbPn87gkwWhdgqAPj2rr/yzuEITncoln9Bgrs/a55ICZRsORJ8Dg9YJC430SSK+rECiOMuiPAFckEGtvZmfL1/P438qBCsIeEeQK+Trfuvid22xZzulIhCo2RLMpQ+sSKjm5WzaGjMqg56MbBZoOlDMrNBIT54hsFafaSF9OkHWQzAvz0jml2Wt4r3KLJkV3+CvSOimSGFEwW/ptpnJgMt2Wi+WiyH4MU2e2RA1GpebYul9XCo76ltGmUvZvse85A0pPZm81IBd9MgdrfhsoYXiOCxc5RtJerRXGvHZsHolnWWQHYbM4l8pJkYdseoc7TJXtjlCgDeZfxz3AH00WihgJM8mYF9aC2lj47IREeJw085sVJzBr0JWCpmQZUBeLQnunBVXe

In [34]:
response['messages'][-1].pretty_print() 

================================== Ai Message ==================================

[{'type': 'text', 'text': "I can find friends of Ram, but I cannot greet each of them individually using the available tools. Would you like me to proceed with just finding Ram's friends?", 'extras': {'signature': 'CpUHAY89a1/lPyLIhHiAsODgc99savVE8Kf1JbOiZovhMNIblNQcXwpvD2Lg9Bo3ExDPAZeBCsHCtMIM4S7adMy7zPNZk9EejSW/odX6KJZ+1CLz9ofnWeQWdrRm5t9TEJOtidmR3GBXV0ZGeuhXRf0MiThpweLtyoy6vg4LOuDusMzOzbPn87gkwWhdgqAPj2rr/yzuEITncoln9Bgrs/a55ICZRsORJ8Dg9YJC430SSK+rECiOMuiPAFckEGtvZmfL1/P438qBCsIeEeQK+Trfuvid22xZzulIhCo2RLMpQ+sSKjm5WzaGjMqg56MbBZoOlDMrNBIT54hsFafaSF9OkHWQzAvz0jml2Wt4r3KLJkV3+CvSOimSGFEwW/ptpnJgMt2Wi+WiyH4MU2e2RA1GpebYul9XCo76ltGmUvZvse85A0pPZm81IBd9MgdrfhsoYXiOCxc5RtJerRXGvHZsHolnWWQHYbM4l8pJkYdseoc7TJXtjlCgDeZfxz3AH00WihgJM8mYF9aC2lj47IREeJw085sVJzBr0JWCpmQZUBeLQnunBVXehIMZ3NeETTR4yGnfObcKvGtprI8Sg2N1x/xpBSrWqg5wbVSIBq27a7kFy1DAFv3PF9VvalMZe4IH+Jlvn0aJKpA0DEtSZH4uhta95z7fhJ6csxnvhYtQu00WmbXegUudUONDPcN